### https://www.kaggle.com/competitions/drawing-with-llms

In [2]:
import kagglehub
import pandas as pd

In [3]:
import concurrent
import io
import logging
import re
import re2

import cairosvg
import kagglehub
import torch
from lxml import etree
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

svg_constraints = kagglehub.package_import('metric/svg-constraints')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('DEVICE', DEVICE)

class SVGSanitizer:
    def __init__(self, constraints, default_svg):
        self.constraints = constraints
        self.default_svg = default_svg
    
    def enforce_constraints(self, svg_string: str) -> str:
        """Enforces constraints on an SVG string, removing disallowed elements
        and attributes.

        Parameters
        ----------
        svg_string : str
            The SVG string to process.

        Returns
        -------
        str
            The processed SVG string, or the default SVG if constraints
            cannot be satisfied.
        """
        logging.info('Sanitizing SVG...')

        try:
            parser = etree.XMLParser(remove_blank_text=True, remove_comments=True)
            root = etree.fromstring(svg_string, parser=parser)
        except etree.ParseError as e:
            logging.error('SVG Parse Error: %s. Returning default SVG.', e)
            return self.default_svg
    
        elements_to_remove = []
        for element in root.iter():
            try:
                tag_name = etree.QName(element.tag).localname
            except ValueError as e:
                return self.default_svg
    
            # Remove disallowed elements
            if tag_name not in self.constraints.allowed_elements:
                elements_to_remove.append(element)
                continue  # Skip attribute checks for removed elements
    
            # Remove disallowed attributes
            attrs_to_remove = []
            for attr in element.attrib:
                attr_name = etree.QName(attr).localname
                if (
                    attr_name
                    not in self.constraints.allowed_elements[tag_name]
                    and attr_name
                    not in self.constraints.allowed_elements['common']
                ):
                    attrs_to_remove.append(attr)
    
            for attr in attrs_to_remove:
                logging.debug(
                    'Attribute "%s" for element "%s" not allowed. Removing.',
                    attr,
                    tag_name,
                )
                del element.attrib[attr]
    
            # Check and remove invalid href attributes
            for attr, value in element.attrib.items():
                 if etree.QName(attr).localname == 'href' and not value.startswith('#'):
                    logging.debug(
                        'Removing invalid href attribute in element "%s".', tag_name
                    )
                    del element.attrib[attr]

            # Validate path elements to help ensure SVG conversion
            if tag_name == 'path':
                d_attribute = element.get('d')
                if not d_attribute:
                    logging.warning('Path element is missing "d" attribute. Removing path.')
                    elements_to_remove.append(element)
                    continue # Skip further checks for this removed element
                # Use regex to validate 'd' attribute format
                path_regex = re2.compile(
                    r'^'  # Start of string
                    r'(?:'  # Non-capturing group for each command + numbers block
                    r'[MmZzLlHhVvCcSsQqTtAa]'  # Valid SVG path commands (adjusted to exclude extra letters)
                    r'\s*'  # Optional whitespace after command
                    r'(?:'  # Non-capturing group for optional numbers
                    r'-?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?'  # First number
                    r'(?:[\s,]+-?\d+(?:\.\d+)?(?:[Ee][+-]?\d+)?)*'  # Subsequent numbers with mandatory separator(s)
                    r')?'  # Numbers are optional (e.g. for Z command)
                    r'\s*'  # Optional whitespace after numbers/command block
                    r')+'  # One or more command blocks
                    r'\s*'  # Optional trailing whitespace
                    r'$'  # End of string
                )
                if not path_regex.match(d_attribute):
                    logging.warning(
                        'Path element has malformed "d" attribute format. Removing path.'
                    )
                    elements_to_remove.append(element)
                    continue
                logging.debug('Path element "d" attribute validated (regex check).')
        
        # Remove elements marked for removal
        for element in elements_to_remove:
            if element.getparent() is not None:
                element.getparent().remove(element)
                logging.debug('Removed element: %s', element.tag)

        try:
            cleaned_svg_string = etree.tostring(root, encoding='unicode')
            return cleaned_svg_string
        except ValueError as e:
            logging.error(
                'SVG could not be sanitized to meet constraints: %s', e
            )
            return self.default_svg

class SVGProcessor:
    @staticmethod
    def clean_and_extract_svgs(text, default_svg):
        text = re.sub(r'^.*?(<svg\b)', r'\1', text, flags=re.DOTALL)
        svg_blocks = re.findall(r'<svg\b.*?</svg>', text, re.DOTALL)
    
        if svg_blocks:
            tmp = re.findall(r'<svg\b.*?', svg_blocks[-1], re.DOTALL)
            if len(tmp) > 1:
                tmp2 = svg_blocks[-1].split('<svg')
                return '<svg ' + tmp2[-1]
            else:
                return svg_blocks[-1]
        else:
            if "<svg" in text and "</svg>" not in text:
                text += "</svg>"
                return text
            return default_svg
    
    @staticmethod
    def svg_conversion_check(topic, base_svg_code, default_svg):
        try:
            cairosvg.svg2png(bytestring=base_svg_code.encode('utf-8'), write_to="temp.png")
            return base_svg_code
        except Exception as e:
            print(f"Failed to convert {topic} due to {str(e)}, Returning default SVG.")
            return default_svg


class Model:
    def __init__(self):
        self.model_path = './lora/lora_16bit_merged_3b_r64_s1000_i1000_v1/' 
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_path)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_path,
            torch_dtype=torch.float16,
            device_map="auto"
        )
        self.model.eval()
        
        self.default_svg = """<svg width="256" height="256" viewBox="0 0 256 256"><circle cx="50" cy="50" r="40" fill="red" /></svg>"""
        self.constraints = svg_constraints.SVGConstraints()
        self.sanitizer = SVGSanitizer(self.constraints, self.default_svg)
    
    def get_response(self, description):
        alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
    
        ### Instruction:
        Please write an SVG code for the given topic?
    
        ### Input:
        {}
    
        ### Response:
        """
        formatted_input = alpaca_prompt.format(description)
        inputs = self.tokenizer([formatted_input], return_tensors="pt").to(DEVICE)
        outputs = self.model.generate(**inputs, max_new_tokens=1024, use_cache=True)
        return self.tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    def predict(self, description: str,base_svg_code: str, max_new_tokens=1024) -> str:
        #output_decoded = self.get_response(description)
        base_svg_code = SVGProcessor.clean_and_extract_svgs(base_svg_code, self.default_svg)
        clean_svg_code = self.sanitizer.enforce_constraints(base_svg_code)
        # SVGProcessor.svg_conversion_check(description, clean_svg_code, self.default_svg)
        
        return clean_svg_code



This code could modify your python environment or operating system.

Review this code at https://www.kaggle.com/code/metric/svg-constraints/versions/1
or in your download cache at /home/vino/.cache/kagglehub/notebooks/metric/svg-constraints/output/versions/1

It is strongly recommended that you run this code within a container
such as Docker to provide a secure, isolated execution environment.
See https://www.kaggle.com/docs/packages for more information.

Do you want to proceed? (y)es/[no]:  y


DEVICE cuda


In [4]:
model=Model()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
# model.predict('sun rising in the east',\
#               df['Svg'].iloc[2136])

In [6]:
import pandas as pd
df=pd.read_csv('response_2_1062.csv',header=[0])
print(df.shape)
df.head(2)

(1062, 5)


,description,svg_code,response,score,response_2
0,"'Golden wheat fields under a setting sun',","<svg viewBox=""0 0 200 200"" width=""200"" height=...","```svg\n<svg viewBox=""0 0 200 200"" width=""200""...",0.994973,"Here's an improved SVG representation of ""Gold..."
1,"'Snowy mountains under a clear blue sky',","<svg viewBox=""0 0 200 100"" width=""200"" height=...","```svg\n<svg viewBox=""0 0 200 100"" width=""200""...",0.978844,"Here's an improved SVG representation of ""Snow..."


In [7]:
import warnings
import logging
from tqdm import tqdm

# Suppress warnings
warnings.simplefilter("ignore")

# Suppress logging messages
logging.getLogger().setLevel(logging.CRITICAL)
tqdm.pandas()

print(df.shape)
df['gpt_svg_2'] = df.progress_apply(lambda x: model.predict(x['description'], x['response_2']), axis=1)

(1062, 5)


100%|█████████████████████████████████████| 1062/1062 [00:00<00:00, 4279.30it/s]


In [8]:
df.to_csv('response_2_1062_clean_svg.csv',index=False)

In [9]:
df['gpt_svg_2'].iloc[0]

'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 200 200" width="200" height="200"><defs><linearGradient id="skyGradient" x1="0" y1="0" x2="0" y2="1"><stop offset="0%" stop-color="#FF7F50"/><stop offset="100%" stop-color="#FF4500"/></linearGradient><radialGradient id="sunGlow" cx="0.5" cy="0.5" r="0.5" fx="0.25" fy="0.25"><stop offset="0%" stop-color="#FFD700"/><stop offset="100%" stop-color="#FF8C00" stop-opacity="0.7"/></radialGradient></defs><rect x="0" y="0" width="200" height="100" fill="url(#skyGradient)"/><circle cx="100" cy="50" r="35" fill="url(#sunGlow)"/><circle cx="100" cy="50" r="25" fill="#FFD700"/><rect x="0" y="100" width="200" height="100" fill="#DAA520"/><g stroke="#8B4513" stroke-width="1.5"><line x1="20" y1="100" x2="20" y2="160"/><line x1="40" y1="100" x2="40" y2="155"/><line x1="60" y1="100" x2="60" y2="165"/><line x1="80" y1="100" x2="80" y2="150"/><line x1="100" y1="100" x2="100" y2="158"/><line x1="120" y1="100" x2="120" y2="162"/><line x1="140" y1="100" x